In [ ]:
"""
================================================================================
stage1_2_bull_faithful.py
--------------------------------------------------------------------------------
FAITHFUL reproduction of Bull (2025), "Neurons as Autoencoders".

This is the version that learns the paper ON ITS OWN TERMS. The single most
important thing to understand before writing any code:

    *** BULL DOES NOT USE BACKPROPAGATION / GRADIENT DESCENT. ***

His title's "backpropagation" is the BIOLOGICAL backward echo up the dendrite,
not the ML algorithm. The weights are trained by a RANDOM HILL-CLIMBER:
    "a learning cycle consists of one randomly chosen weight from either the
     hidden layer or output layer being adjusted by a random amount from the
     range [-R, +R]" ... "the new network configuration is chosen if its mean
     squared error is reduced in comparison to that of the current solution,
     with ties broken at random."

So there are NO gradients here at all. We store weights as plain numpy arrays
and mutate one number at a time. This file deliberately uses numpy, not torch,
because gradients are exactly what we are NOT using — bringing in autograd would
misrepresent the paper. (Stage 3, in the other file, does the gradient version.)

Three network types are built, matching the paper's Figures 3-4:
    NN  : plain MLP, supervised only           (baseline)
    ANN : one shared decoder LAYER on the hidden layer (standard autoencoder)
    NAN : each hidden NEURON has its OWN decoder (the paper's proposal)

Data comes from the NK model — a *tuneable* regression source (knobs N and K).

Paper settings reproduced: H=10, R=1.0, 10,000 iterations, 20 runs averaged,
train/test = 1000 examples each, inputs use -1.0 in place of 0.

Author scaffolding: Jamal Idrissou
================================================================================
"""

import numpy as np


# ==============================================================================
# STAGE 1 (LEARN + APPLY): the NK model as a tuneable data source
# ==============================================================================
class NKModel:
    """
    Kauffman's NK model, used here as a regression function (after Bull 2022).

    Why this and not MNIST? Because it has two difficulty knobs:
        N = number of binary features (genes)
        K = how many OTHER genes each gene's fitness depends on (epistasis)
    Raising K makes the input->output mapping more rugged / less correlated,
    so you can deliberately place a problem in the "easy" or "hard" regime and
    see WHERE neuron-autoencoding helps. That conditional finding is the whole
    point of the paper, and you can't get it from a single fixed dataset.

    Mechanism: each gene i has its own lookup table over its own value plus the
    values of its K interaction partners (2^(K+1) rows of random fitness in
    [0,1)). The genome fitness is the mean of the N per-gene table lookups.
    """

    def __init__(self, N, K, seed=0):
        self.N = N
        self.K = K
        rng = np.random.default_rng(seed)

        # For each gene, pick which K OTHER genes it depends on (its partners).
        # Gene i always depends on itself plus K distinct others -> (K+1) inputs.
        self.partners = []
        for i in range(N):
            others = [j for j in range(N) if j != i]
            chosen = rng.choice(others, size=K, replace=False) if K > 0 else np.array([], dtype=int)
            # store [self, partner1, ... partnerK]; order matters for table indexing
            self.partners.append(np.concatenate(([i], chosen)).astype(int))

        # For each gene, a random fitness table with 2^(K+1) rows (one per
        # combination of the gene + its K partners). Values in [0, 1).
        self.tables = [rng.random(2 ** (K + 1)) for _ in range(N)]

    def fitness(self, genome):
        """
        genome : array of N values in {0,1}. Returns scalar fitness in [0,1).
        For each gene we read off its (K+1) relevant bits, interpret them as a
        binary index into that gene's table, and average the N looked-up values.
        """
        total = 0.0
        for i in range(self.N):
            bits = genome[self.partners[i]]                 # the (K+1) relevant bits
            idx = int("".join(str(int(b)) for b in bits), 2)  # bits -> table row index
            total += self.tables[i][idx]
        return total / self.N

    def make_dataset(self, n_samples, seed=0):
        """
        Build (X, y). X is {-1, +1} encoded (paper replaces 0 with -1.0 as a
        neural-net input convention); y is the scalar NK fitness in [0,1).
        Returns X: (n_samples, N) float, y: (n_samples, 1) float.
        """
        rng = np.random.default_rng(seed)
        genomes = rng.integers(0, 2, size=(n_samples, self.N))   # {0,1}
        y = np.array([[self.fitness(g)] for g in genomes])        # (n,1) in [0,1)
        X = np.where(genomes == 0, -1.0, 1.0)                     # {0,1} -> {-1,+1}
        return X.astype(np.float64), y.astype(np.float64)


# ------------------------------------------------------------------------------
# small helpers
# ------------------------------------------------------------------------------
def sigmoid(z):
    # paper uses a sigmoid transfer function on all nodes (not linear)
    return 1.0 / (1.0 + np.exp(-z))


# ==============================================================================
# STAGE 2 (APPLY): the three network types, trained by random hill-climbing
# ==============================================================================
class NeuronAutoencoderNetwork:
    """
    A two-layer MLP (N -> H -> 1) where, optionally, each hidden neuron also
    carries its own decoder (N weights + bias) that tries to rebuild the input
    from that single neuron's scalar activation. Selecting the `mode` switches
    between the paper's three architectures:

        mode="NN"  : plain MLP, supervised hill-climbing only.
        mode="ANN" : a SINGLE decoder layer hangs off the whole hidden layer
                     (H -> N). Standard "one decoder layer" autoencoder.
        mode="NAN" : EACH of the H hidden neurons has its OWN 1->N decoder.
                     This is the paper's neuron-autoencoder proposal.

    All weights are plain numpy arrays. Training mutates ONE weight per cycle
    and keeps the change only if MSE improves (hill-climbing, ties random).
    """

    def __init__(self, N, H, mode="NAN", R=1.0, seed=0):
        self.N, self.H, self.mode, self.R = N, H, mode, R
        rng = np.random.default_rng(seed)
        self.rng = rng

        # --- forward weights, seeded in [-1, 1] as in the paper --------------
        # hidden layer: maps N inputs -> H hidden units (+ bias per unit)
        self.W1 = rng.uniform(-1, 1, size=(N, H))
        self.b1 = rng.uniform(-1, 1, size=(H,))
        # output layer: maps H hidden -> 1 output (+ bias)
        self.W2 = rng.uniform(-1, 1, size=(H, 1))
        self.b2 = rng.uniform(-1, 1, size=(1,))

        # --- decoder weights depending on mode -------------------------------
        if mode == "NAN":
            # one decoder PER hidden neuron: each maps its scalar a_j -> N-dim.
            # Dweight[j] is neuron j's (N,) decode vector; Dbias[j] its (N,) bias.
            self.Dweight = rng.uniform(-1, 1, size=(H, N))
            self.Dbias   = rng.uniform(-1, 1, size=(H, N))
        elif mode == "ANN":
            # one shared decoder LAYER: hidden vector (H,) -> reconstruct (N,).
            self.Dw = rng.uniform(-1, 1, size=(H, N))
            self.Db = rng.uniform(-1, 1, size=(N,))
        # mode == "NN" has no decoder at all.

    # --------------------------------------------------------------------------
    def hidden(self, X):
        """Hidden activations a = sigmoid(X W1 + b1). X:(B,N) -> (B,H)."""
        return sigmoid(X @ self.W1 + self.b1)

    def forward(self, X):
        """Full task forward pass -> output in (B,1) via sigmoid output node."""
        a = self.hidden(X)
        return sigmoid(a @ self.W2 + self.b2)

    # --------------------------------------------------------------------------
    def task_mse(self, X, y):
        """Supervised objective: MSE between network output and target y."""
        return float(np.mean((self.forward(X) - y) ** 2))

    def autoenc_mse(self, X):
        """
        Reconstruction objective. For NAN this is the MEAN over the H neurons of
        each neuron's own reconstruction MSE (matches the paper, where 'the NAN
        autoencoder error is the average of the H hidden layer nodes').
        For ANN it is the single shared-decoder reconstruction MSE.
        """
        a = self.hidden(X)                       # (B, H) hidden activations
        if self.mode == "NAN":
            # neuron j reconstructs X from its scalar a[:,j]:
            #   xhat[:, j, :] = a[:,j,None] * Dweight[j] + Dbias[j]
            # build (B, H, N) then compare against X broadcast to (B,1,N).
            xhat = a[:, :, None] * self.Dweight[None, :, :] + self.Dbias[None, :, :]
            target = X[:, None, :]               # (B,1,N) -> broadcasts to (B,H,N)
            return float(np.mean((xhat - target) ** 2))
        elif self.mode == "ANN":
            xhat = sigmoid(a @ self.Dw + self.Db)  # (B,N) single decoder layer
            # NOTE: inputs are in {-1,1}; we compare in raw space as the paper
            # measures MSE on the input values directly.
            return float(np.mean((xhat - X) ** 2))
        else:
            return 0.0  # NN has no autoencoder

    # --------------------------------------------------------------------------
    def _perturb(self, arr, idx):
        """Apply a random [-R,+R] nudge to one entry; return the OLD value so
        the caller can revert if the move is not kept (hill-climbing)."""
        old = arr[idx]
        arr[idx] = old + self.rng.uniform(-self.R, self.R)
        return old

    def train(self, X, y, iterations=10000, record_every=200):
        """
        Bull's training loop. Each iteration:
          1. Decide cycle type. For NN every cycle is a task cycle. For ANN/NAN
             it is a task cycle or an autoencoding cycle with probability 0.5.
          2. Pick ONE weight in the relevant group, perturb it by [-R,+R].
          3. Recompute the relevant error. Keep the change only if it did not
             get worse (improvement kept; ties broken at random by a coin flip).
        Returns a history dict of task / autoencoder MSE sampled over time.
        """
        hist = {"iter": [], "task_mse": [], "ae_mse": []}

        for it in range(iterations):
            # ---- 1. choose cycle type -------------------------------------
            is_ae_cycle = (self.mode != "NN") and (self.rng.random() < 0.5)

            if is_ae_cycle:
                # ---- AUTOENCODING CYCLE: judged by reconstruction MSE ------
                before = self.autoenc_mse(X)
                if self.mode == "NAN":
                    # perturb one decoder weight OR the hidden weight feeding a
                    # neuron — i.e. any weight that affects reconstruction. The
                    # paper adjusts a hidden-layer-associated weight; here we
                    # nudge a single decoder weight (the neuron's own AE param).
                    j = self.rng.integers(self.H)
                    k = self.rng.integers(self.N)
                    old = self._perturb(self.Dweight, (j, k))
                    after = self.autoenc_mse(X)
                    if after > before:                 # worse -> revert
                        self.Dweight[j, k] = old
                else:  # ANN
                    r = self.rng.integers(self.H); c = self.rng.integers(self.N)
                    old = self._perturb(self.Dw, (r, c))
                    after = self.autoenc_mse(X)
                    if after > before:
                        self.Dw[r, c] = old
            else:
                # ---- TASK CYCLE: judged by supervised task MSE -------------
                before = self.task_mse(X, y)
                # pick a weight from the forward path (hidden or output layer)
                if self.rng.random() < 0.5:
                    j = self.rng.integers(self.H); k = self.rng.integers(self.N)
                    old = self._perturb(self.W1, (k, j))
                    after = self.task_mse(X, y)
                    if after > before:
                        self.W1[k, j] = old
                else:
                    j = self.rng.integers(self.H)
                    old = self._perturb(self.W2, (j, 0))
                    after = self.task_mse(X, y)
                    if after > before:
                        self.W2[j, 0] = old

            # ---- record progress periodically -----------------------------
            if it % record_every == 0:
                hist["iter"].append(it)
                hist["task_mse"].append(self.task_mse(X, y))
                hist["ae_mse"].append(self.autoenc_mse(X))

        return hist


# ==============================================================================
# STAGE 2 RUNNER (EXPERIMENT): compare NN / ANN / NAN, sweep N and K
# ==============================================================================
if __name__ == "__main__":
    # --- a quick single comparison so you can see it run ---------------------
    N, K, H = 20, 5, 10
    nk = NKModel(N=N, K=K, seed=1)
    Xtr, ytr = nk.make_dataset(1000, seed=10)
    Xte, yte = nk.make_dataset(1000, seed=20)
    print(f"NK data: N={N} K={K} | train {Xtr.shape} test {Xte.shape}")

    ITERS = 4000  # paper uses 10000; smaller here so the demo is quick
    results = {}
    for mode in ["NN", "ANN", "NAN"]:
        net = NeuronAutoencoderNetwork(N=N, H=H, mode=mode, R=1.0, seed=2)
        net.train(Xtr, ytr, iterations=ITERS, record_every=500)
        results[mode] = net.task_mse(Xte, yte)   # held-out task error
        print(f"  {mode:3s}  test task MSE = {results[mode]:.5f}")

    print("\nReproduce the paper's *conditional* finding by sweeping:")
    print("  small N (e.g. 20) vs large N (e.g. 200+), K in {0,5,10,15},")
    print("  20 runs each, average, and run a t-test (NAN vs ANN, NAN vs NN).")

NK data: N=20 K=5 | train (1000, 20) test (1000, 20)
  NN   test task MSE = 0.00402
  ANN  test task MSE = 0.00479
  NAN  test task MSE = 0.00694

Reproduce the paper's *conditional* finding by sweeping:
  small N (e.g. 20) vs large N (e.g. 200+), K in {0,5,10,15},
  20 runs each, average, and run a t-test (NAN vs ANN, NAN vs NN).


In [ ]:
"""
================================================================================
stage3_4_gradient_and_spiking.py
--------------------------------------------------------------------------------
STAGE 3: the gradient-descent reinterpretation of Bull's idea (scales to real
         data, connects to your existing MNIST repo).
STAGE 4: extending neuron-level local autoencoding to SPIKING neural networks
         (SNNs), which is the integration your internship asked for.

Read stage1_2_bull_faithful.py FIRST. That file is the paper as written
(hill-climbing, no gradients). THIS file is the practical engineering version:
we keep Bull's *structure* (each neuron reconstructs its own input, training
alternates task vs autoencoding cycles) but swap his random hill-climber for
gradient descent so it scales beyond tiny N and runs on GPUs.

WHY BOTH VERSIONS MATTER (the "how to learn how to learn" answer):
  - The faithful version teaches you what the paper actually claims and why the
    NK tuneability matters. You cannot critique or extend a method you have only
    seen through a reframing.
  - The gradient version is what you will actually experiment with at scale and
    what plugs into your skin-cancer-era PyTorch skills.
Showing your supervisor that you understand the difference is itself a result.
================================================================================
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


# ==============================================================================
# STAGE 3 (APPLY): gradient-based neuron-autoencoder layer
# ==============================================================================
class NeuronAELayer(nn.Module):
    """
    Gradient-trained version of Bull's neuron-autoencoder. Identical idea to the
    faithful file, but the weights are nn.Parameters so autograd can train them.

    Each of the `out_features` neurons:
        encode : x (I,) -> a_j (scalar)         [the ordinary forward synapse]
        decode : a_j    -> x_hat_j (I,)         [its own dendritic 'echo']
        local loss_j = MSE(x_hat_j, x)

    Shapes (B batch, I in, N neurons): x (B,I) -> a (B,N), x_hat (B,N,I).
    """

    def __init__(self, in_features, out_features):
        super().__init__()
        self.encoder = nn.Linear(in_features, out_features)      # forward synapse
        # per-neuron decoder: neuron j maps its scalar -> I-dim reconstruction
        self.dec_w = nn.Parameter(torch.empty(out_features, in_features))
        self.dec_b = nn.Parameter(torch.zeros(out_features, in_features))
        nn.init.kaiming_uniform_(self.dec_w, a=5 ** 0.5)

    def forward(self, x):
        # soma signal each neuron passes forward (this is the normal layer out)
        return torch.sigmoid(self.encoder(x))

    def local_loss(self, x):
        a = torch.sigmoid(self.encoder(x))                       # (B,N) codes
        # outer-product style broadcast -> one reconstruction per neuron
        x_hat = a.unsqueeze(-1) * self.dec_w.unsqueeze(0) + self.dec_b.unsqueeze(0)
        return F.mse_loss(x_hat, x.unsqueeze(1).expand_as(x_hat))


class GradientNAN(nn.Module):
    """A small regression MLP whose hidden layer is a NeuronAELayer.
    Returns the task prediction, and on request the hidden layer's local loss."""

    def __init__(self, in_dim, hidden=10, out_dim=1):
        super().__init__()
        self.hidden = NeuronAELayer(in_dim, hidden)
        self.head = nn.Linear(hidden, out_dim)

    def forward(self, x, want_local=False):
        local = self.hidden.local_loss(x) if want_local else None
        a = self.hidden(x)
        y = torch.sigmoid(self.head(a))      # NK targets are in [0,1)
        return (y, local) if want_local else y


def train_gradient_nan(model, X, y, steps=3000, local_weight=1.0):
    """
    Alternating gradient version of Bull's two cycle types:
        task cycle  -> minimise supervised MSE (head + encoder get gradients)
        local cycle -> minimise neuron reconstruction MSE (encoder + decoder)
    We alternate on a coin flip per step, mirroring his 0.5 probability, but use
    one Adam optimiser and real gradients instead of single-weight hill-climbing.
    """
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    for step in range(steps):
        if torch.rand(1).item() < 0.5:        # ---- task cycle ----
            opt.zero_grad()
            pred = model(X)
            loss = F.mse_loss(pred, y)
            loss.backward(); opt.step()
        else:                                  # ---- autoencoding cycle ----
            opt.zero_grad()
            _, local = model(X, want_local=True)
            (local_weight * local).backward(); opt.step()
    with torch.no_grad():
        return F.mse_loss(model(X), y).item()


# ==============================================================================
# STAGE 4 (LEARN -> APPLY -> EXPERIMENT): neuron-AE inside a SPIKING network
# ==============================================================================
"""
WHY SNNs FIT BULL'S IDEA UNUSUALLY WELL
---------------------------------------
Bull motivates neuron-AE biologically: a real neuron sends a forward spike down
its axon and a backward "echo" up its dendrites. SNNs are the ANN family that
actually models neurons as spiking units over time, so neuron-level local
learning is a more natural fit here than in a plain MLP. Two honest challenges:

  (1) Spikes are non-differentiable (a threshold crossing is a step function),
      so standard backprop doesn't apply directly. The field's standard fix is
      the SURROGATE GRADIENT: use the hard threshold on the forward pass but a
      smooth function's derivative on the backward pass. (See sources below.)

  (2) "Reconstruct the input" must be defined over TIME. A spiking neuron's
      output is a spike train, and its natural scalar summary is its firing
      rate (spikes / timesteps). So our neuron-AE decodes from that rate.

This makes the SNN neuron-AE a clean research question: does forcing each
spiking neuron to retain enough information to reconstruct its input (via its
firing rate) shape better representations, exactly as Bull found for MLPs?
"""


class SurrogateSpike(torch.autograd.Function):
    """
    Leaky-integrate-and-fire spike with a surrogate gradient.

    forward : hard threshold. Output is 1.0 if membrane potential v >= 0 (we
              pass v already shifted by the threshold), else 0.0 — a real spike.
    backward: pretend the step was a smooth fast-sigmoid so a usable gradient
              flows. This is the standard surrogate-gradient trick that makes
              SNNs trainable with autograd. grad ~ 1/(1+|v|*scale)^2.
    """
    @staticmethod
    def forward(ctx, v):
        ctx.save_for_backward(v)
        return (v >= 0).float()

    @staticmethod
    def backward(ctx, grad_output):
        (v,) = ctx.saved_tensors
        scale = 10.0
        sg = 1.0 / (scale * torch.abs(v) + 1.0) ** 2   # surrogate derivative
        return grad_output * sg


spike_fn = SurrogateSpike.apply


class SpikingNeuronAELayer(nn.Module):
    """
    A layer of leaky-integrate-and-fire (LIF) neurons, where each neuron ALSO
    autoencodes the (time-averaged) input from its OWN firing rate.

    Over T timesteps each neuron:
        - integrates input current into a leaky membrane potential v,
        - emits a spike when v crosses threshold, then resets,
        - its firing RATE (mean spikes over T) is its scalar 'code' a_j,
        - a per-neuron decoder maps a_j -> reconstruction of the mean input.

    So this is Bull's neuron-AE with the scalar bottleneck being the neuron's
    firing rate rather than a static activation — the temporal analogue.
    """

    def __init__(self, in_features, out_features, T=20, beta=0.9, thresh=1.0):
        super().__init__()
        self.in_features, self.out_features = in_features, out_features
        self.T = T            # number of simulation timesteps
        self.beta = beta      # membrane leak (0=no memory, 1=perfect integrator)
        self.thresh = thresh  # firing threshold

        self.fc = nn.Linear(in_features, out_features)   # input -> current
        # per-neuron decoder from firing rate (scalar) back to the input vector
        self.dec_w = nn.Parameter(torch.empty(out_features, in_features))
        self.dec_b = nn.Parameter(torch.zeros(out_features, in_features))
        nn.init.kaiming_uniform_(self.dec_w, a=5 ** 0.5)

    def simulate(self, x):
        """
        Run the LIF dynamics for T steps with a constant input current derived
        from x (rate-coding the static input). Returns:
            rate    : (B, N) mean firing rate per neuron  (the AE 'code')
            spikes  : (B, T, N) full spike train (kept for inspection/plots)
        """
        B = x.size(0)
        current = self.fc(x)                       # (B, N) constant drive
        v = torch.zeros(B, self.out_features, device=x.device)
        spike_sum = torch.zeros(B, self.out_features, device=x.device)
        spikes = []
        for _ in range(self.T):
            v = self.beta * v + current            # leaky integration
            s = spike_fn(v - self.thresh)          # spike if v >= thresh
            v = v * (1.0 - s)                      # hard reset where a spike fired
            spike_sum = spike_sum + s
            spikes.append(s)
        rate = spike_sum / self.T                  # (B, N) firing rate in [0,1]
        return rate, torch.stack(spikes, dim=1)

    def forward(self, x):
        rate, _ = self.simulate(x)
        return rate                                # passed forward to next layer

    def local_loss(self, x):
        """Neuron-AE loss: each neuron rebuilds the input from its firing rate."""
        rate, _ = self.simulate(x)                 # (B, N)
        x_hat = rate.unsqueeze(-1) * self.dec_w.unsqueeze(0) + self.dec_b.unsqueeze(0)
        return F.mse_loss(x_hat, x.unsqueeze(1).expand_as(x_hat))


class SpikingNAN(nn.Module):
    """Spiking regression network with a neuron-autoencoding hidden layer."""

    def __init__(self, in_dim, hidden=10, out_dim=1, T=20):
        super().__init__()
        self.hidden = SpikingNeuronAELayer(in_dim, hidden, T=T)
        self.head = nn.Linear(hidden, out_dim)

    def forward(self, x, want_local=False):
        local = self.hidden.local_loss(x) if want_local else None
        rate = self.hidden(x)            # firing rates -> readout
        y = torch.sigmoid(self.head(rate))
        return (y, local) if want_local else y


# ==============================================================================
# RUNNER: confirm both stages train on a toy regression target
# ==============================================================================
if __name__ == "__main__":
    torch.manual_seed(0)
    # toy data standing in for the NK regression (import NKModel from stage 1/2
    # for the real thing). Inputs in {-1,1}, target in [0,1).
    B, I = 256, 20
    X = (torch.rand(B, I) > 0.5).float() * 2 - 1
    y = torch.sigmoid(X.sum(dim=1, keepdim=True) * 0.3)   # smooth-ish target

    print("=== STAGE 3: gradient neuron-AE (NAN) ===")
    g = GradientNAN(in_dim=I, hidden=10)
    mse_g = train_gradient_nan(g, X, y, steps=1500, local_weight=1.0)
    print(f"  final task MSE: {mse_g:.5f}")

    print("\n=== STAGE 4: spiking neuron-AE (Spiking NAN) ===")
    s = SpikingNAN(in_dim=I, hidden=10, T=20)
    opt = torch.optim.Adam(s.parameters(), lr=1e-2)
    for step in range(800):
        if torch.rand(1).item() < 0.5:                  # task cycle
            opt.zero_grad(); F.mse_loss(s(X), y).backward(); opt.step()
        else:                                            # autoencoding cycle
            opt.zero_grad(); s(X, want_local=True)[1].backward(); opt.step()
    with torch.no_grad():
        print(f"  final task MSE: {F.mse_loss(s(X), y).item():.5f}")
        rate, spk = s.hidden.simulate(X)
        print(f"  mean firing rate: {rate.mean().item():.3f} "
              f"(spike train shape {tuple(spk.shape)} = B x T x neurons)")
    print("\nExperiment next: sweep T (timesteps), beta (leak), and local_weight;")
    print("compare Spiking NAN vs a spiking baseline with no neuron-AE, NK-style.")

=== STAGE 3: gradient neuron-AE (NAN) ===
  final task MSE: 0.00105

=== STAGE 4: spiking neuron-AE (Spiking NAN) ===
  final task MSE: 0.00224
  mean firing rate: 0.416 (spike train shape (256, 20, 10) = B x T x neurons)

Experiment next: sweep T (timesteps), beta (leak), and local_weight;
compare Spiking NAN vs a spiking baseline with no neuron-AE, NK-style.


In [ ]:
"""
================================================================================
stage5_mnist.py
--------------------------------------------------------------------------------
Porting Bull's neuron-autoencoder from the NK regression task to MNIST
classification — BOTH a gradient-only version and a gradient+spiking version.

Read stage1_2_bull_faithful.py and stage3_4_gradient_and_spiking.py first.
This file deliberately REUSES those building blocks unchanged wherever possible;
the comments call out exactly what changed from NK and what did not.

WHAT CHANGES GOING FROM NK -> MNIST (and what doesn't)
------------------------------------------------------
  (1) Task: regression (1 sigmoid node, MSE) -> classification (10 logits,
      cross-entropy). This touches ONLY the head and the task-loss line.
  (2) Reconstruction target: a 20-dim {-1,+1} vector -> a (resolution^2)-dim
      pixel vector in [0,1]. The decoder is mechanically identical (scalar ->
      input_dim); only the dimension and the value range change.
  (3) Difficulty knob: NK gave us N (size) and K (ruggedness) for free. MNIST is
      ONE fixed problem, so to reproduce Bull's *conditional* finding we
      MANUFACTURE the sweep axes:
          NK's N (problem size)    -> input RESOLUTION (7x7, 14x14, 28x28)
          NK's K (interdependence) -> task HARDNESS proxy (#classes / noise)
      That parallel is the whole point: the MNIST study has the same 2D shape as
      the NK Figure-6 study, so the two sit side by side in your write-up.

The neuron-AE MECHANISM itself (each neuron reconstructs its own input from its
single scalar code; training alternates task vs autoencoding cycles) is UNCHANGED
from the NK files. That invariance is the thing to understand.

Author scaffolding: Jamal Idrissou
================================================================================
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


# ==============================================================================
# REUSED VERBATIM FROM stage3_4 — the neuron-AE layer does NOT change for MNIST.
# (Copied here so this file runs standalone; in your repo, import it instead:
#     from stage3_4_gradient_and_spiking import NeuronAELayer, SpikingNeuronAELayer)
# ==============================================================================
class NeuronAELayer(nn.Module):
    """Gradient neuron-autoencoder layer. IDENTICAL to the NK version.
    Each of `out_features` neurons encodes x->scalar and decodes scalar->x_hat."""

    def __init__(self, in_features, out_features):
        super().__init__()
        self.encoder = nn.Linear(in_features, out_features)
        self.dec_w = nn.Parameter(torch.empty(out_features, in_features))
        self.dec_b = nn.Parameter(torch.zeros(out_features, in_features))
        nn.init.kaiming_uniform_(self.dec_w, a=5 ** 0.5)

    def forward(self, x):
        return torch.sigmoid(self.encoder(x))           # soma signal -> next layer

    def local_loss(self, x):
        a = torch.sigmoid(self.encoder(x))              # (B,N) per-neuron codes
        x_hat = a.unsqueeze(-1) * self.dec_w.unsqueeze(0) + self.dec_b.unsqueeze(0)
        return F.mse_loss(x_hat, x.unsqueeze(1).expand_as(x_hat))


class SurrogateSpike(torch.autograd.Function):
    """Hard-threshold spike forward, smooth surrogate gradient backward.
    IDENTICAL to stage3_4 — the spiking mechanism is task-agnostic."""
    @staticmethod
    def forward(ctx, v):
        ctx.save_for_backward(v)
        return (v >= 0).float()

    @staticmethod
    def backward(ctx, grad_output):
        (v,) = ctx.saved_tensors
        sg = 1.0 / (10.0 * torch.abs(v) + 1.0) ** 2
        return grad_output * sg


spike_fn = SurrogateSpike.apply


class SpikingNeuronAELayer(nn.Module):
    """LIF layer where each neuron autoencodes the input from its firing RATE.
    IDENTICAL to stage3_4; the firing rate is the temporal analogue of the
    static scalar code used in the gradient version."""

    def __init__(self, in_features, out_features, T=20, beta=0.9, thresh=1.0):
        super().__init__()
        self.out_features, self.T, self.beta, self.thresh = out_features, T, beta, thresh
        self.fc = nn.Linear(in_features, out_features)
        self.dec_w = nn.Parameter(torch.empty(out_features, in_features))
        self.dec_b = nn.Parameter(torch.zeros(out_features, in_features))
        nn.init.kaiming_uniform_(self.dec_w, a=5 ** 0.5)

    def simulate(self, x):
        B = x.size(0)
        current = self.fc(x)
        v = torch.zeros(B, self.out_features, device=x.device)
        spike_sum = torch.zeros(B, self.out_features, device=x.device)
        for _ in range(self.T):
            v = self.beta * v + current
            s = spike_fn(v - self.thresh)
            v = v * (1.0 - s)
            spike_sum = spike_sum + s
        return spike_sum / self.T                        # (B,N) firing rate

    def forward(self, x):
        return self.simulate(x)

    def local_loss(self, x):
        rate = self.simulate(x)
        x_hat = rate.unsqueeze(-1) * self.dec_w.unsqueeze(0) + self.dec_b.unsqueeze(0)
        return F.mse_loss(x_hat, x.unsqueeze(1).expand_as(x_hat))


# ==============================================================================
# THE ONLY GENUINELY NEW CODE: classification heads + MNIST data + training.
# ==============================================================================

# ----- 1. GRADIENT-ONLY MNIST model -------------------------------------------
class GradientNAN_MNIST(nn.Module):
    """
    Neuron-AE MLP for MNIST. The ONLY differences from the NK GradientNAN:
        - head is Linear(hidden, num_classes) instead of Linear(hidden, 1)
        - forward returns raw LOGITS (no output sigmoid); cross-entropy wants logits
    The hidden NeuronAELayer is untouched.
    """

    def __init__(self, in_dim, hidden=256, num_classes=10):
        super().__init__()
        self.hidden = NeuronAELayer(in_dim, hidden)
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, x, want_local=False):
        local = self.hidden.local_loss(x) if want_local else None
        logits = self.head(self.hidden(x))               # raw logits
        return (logits, local) if want_local else logits


# ----- 2. GRADIENT + SPIKING MNIST model --------------------------------------
class SpikingNAN_MNIST(nn.Module):
    """Spiking neuron-AE for MNIST. Same head swap as above; hidden layer is the
    spiking neuron-AE. Readout is a linear map on the firing-rate vector."""

    def __init__(self, in_dim, hidden=256, num_classes=10, T=20):
        super().__init__()
        self.hidden = SpikingNeuronAELayer(in_dim, hidden, T=T)
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, x, want_local=False):
        local = self.hidden.local_loss(x) if want_local else None
        logits = self.head(self.hidden(x))               # logits from firing rates
        return (logits, local) if want_local else logits


# ----- 3. MNIST data with the NK->MNIST knob mapping --------------------------
def get_mnist_loaders(resolution=28, num_classes=10, batch_size=64,
                      noise=0.0, root="./data"):
    """
    Build MNIST train/test loaders implementing the two manufactured sweep axes:

        resolution  -> the N-axis (problem size). 7, 14, or 28. Images are
                       downscaled with transforms.Resize, so in_dim = resolution^2.
        num_classes -> part of the K-proxy (hardness). 2 keeps only digits 0/1
                       (easy); 10 is the full task (hard).
        noise       -> the other half of the K-proxy. Gaussian pixel noise std;
                       higher = more tangled input->label mapping.

    Returns (train_loader, test_loader, in_dim). Falls back to a synthetic
    MNIST-shaped dataset if torchvision can't download.
    """
    from torchvision import datasets, transforms

    tf = transforms.Compose([
        transforms.Resize((resolution, resolution)),     # <-- the N-axis knob
        transforms.ToTensor(),                           # -> [0,1], shape (1,res,res)
    ])

    def _subset_and_flatten(ds):
        # keep only the first `num_classes` digit labels, flatten images
        X, y = [], []
        for img, label in ds:
            if label < num_classes:
                X.append(img.view(-1))                   # (res*res,)
                y.append(label)
        X = torch.stack(X); y = torch.tensor(y)
        if noise > 0:
            X = (X + noise * torch.randn_like(X)).clamp(0, 1)   # K-proxy noise
        return TensorDataset(X, y)

    try:
        tr = datasets.MNIST(root, train=True,  download=True, transform=tf)
        te = datasets.MNIST(root, train=False, download=True, transform=tf)
        train_ds = _subset_and_flatten(tr)
        test_ds  = _subset_and_flatten(te)
    except Exception as e:
        # ---- offline fallback: synthetic data of the right shape -----------
        print(f"[get_mnist_loaders] real MNIST unavailable ({type(e).__name__}); "
              f"using synthetic stand-in so the code still runs.")
        in_dim = resolution * resolution
        def _synth(n):
            y = torch.randint(0, num_classes, (n,))
            # give each class a distinct mean pattern so the task is learnable
            X = torch.rand(n, in_dim) * 0.1 + (y.float().unsqueeze(1) / num_classes)
            X = (X + noise * torch.randn_like(X)).clamp(0, 1)
            return TensorDataset(X, y)
        train_ds, test_ds = _synth(2000), _synth(1000)

    in_dim = resolution * resolution
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)
    return train_loader, test_loader, in_dim


# ----- 4. training loop: same alternating cycle as NK, classification loss ----
def train_mnist(model, train_loader, test_loader, epochs=3,
                local_weight=1.0, device="cpu"):
    """
    Bull's alternating scheme, MNIST edition. Per batch we coin-flip between:
        TASK cycle  : minimise CROSS-ENTROPY on labels  (was MSE for NK)
        LOCAL cycle : minimise neuron reconstruction MSE (UNCHANGED from NK)
    `local_weight=0` recovers a plain classifier baseline for the ablation.
    Returns final test accuracy (%).
    """
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            if local_weight == 0 or torch.rand(1).item() < 0.5:   # ---- task ----
                opt.zero_grad()
                loss = F.cross_entropy(model(images), labels)     # <-- the change
                loss.backward(); opt.step()
            else:                                                # ---- local ----
                opt.zero_grad()
                _, local = model(images, want_local=True)
                (local_weight * local).backward(); opt.step()

    # ---- evaluate ----
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            correct += (model(images).argmax(1) == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total


# ==============================================================================
# 5. SWEEP HARNESS — the MNIST analogue of the NK Figure-6 study
# ==============================================================================
def resolution_difficulty_sweep(model_kind="gradient", runs=20, epochs=2,
                                 device="cpu"):
    """
    Reproduce the SHAPE of Bull's conditional finding on MNIST by sweeping:
        resolution (N-axis)  x  num_classes (K-proxy)
    comparing neuron-AE ON (local_weight=1) vs OFF (local_weight=0), averaged
    over `runs` seeds.
    """
    import numpy as np
    import scipy.stats

    resolutions = [7, 14, 28]      # Added 28 for the full N-axis
    class_counts = [2, 10]         # 2 = easy, 10 = hard (K-proxy)
    Model = GradientNAN_MNIST if model_kind == "gradient" else SpikingNAN_MNIST

    print(f"\n=== {model_kind.upper()} sweep: resolution x difficulty ===")
    print(f"{'res':>4} {'classes':>8} {'AE-off acc':>11} {'AE-on acc':>10} {'delta':>7} {'p-value':>8}")
    table = {}
    for res in resolutions:
        for nc in class_counts:
            accs_off, accs_on = [], []
            for seed in range(runs):
                torch.manual_seed(seed)
                # Lowered batch_size to 64 to prevent OOM errors at 28x28 resolution
                tr, te, in_dim = get_mnist_loaders(resolution=res, num_classes=nc,
                                                   batch_size=64)
                off = train_mnist(Model(in_dim, hidden=128, num_classes=nc),
                                  tr, te, epochs=epochs, local_weight=0.0, device=device)

                torch.manual_seed(seed)   # same init/data ordering for fair pairing
                tr, te, in_dim = get_mnist_loaders(resolution=res, num_classes=nc,
                                                   batch_size=64)
                on = train_mnist(Model(in_dim, hidden=128, num_classes=nc),
                                 tr, te, epochs=epochs, local_weight=1.0, device=device)

                accs_off.append(off)
                accs_on.append(on)

            mo, mn = np.mean(accs_off), np.mean(accs_on)
            # Calculate statistical significance
            _, p_val = scipy.stats.ttest_rel(accs_on, accs_off)
            table[(res, nc)] = (mo, mn, p_val)
            print(f"{res:>4} {nc:>8} {mo:>11.2f} {mn:>10.2f} {mn-mo:>+7.2f} {p_val:>8.4f}")

    print("\nInterpretation guide: a POSITIVE delta that grows with resolution ")
    print("indicates the neuron-autoencoder is beneficial at larger N scales.")
    return table


# ==============================================================================
# RUNNER
# ==============================================================================
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device: {device}")

    # ---- single-shot sanity check of both models -----------------------------
    # Reduced batch size to 64 here as well for consistency and memory safety
    tr, te, in_dim = get_mnist_loaders(resolution=14, num_classes=10, batch_size=64)

    print("\n=== GRADIENT-ONLY neuron-AE on MNIST ===")
    g = GradientNAN_MNIST(in_dim, hidden=128, num_classes=10)
    acc_g = train_mnist(g, tr, te, epochs=2, local_weight=1.0, device=device)
    print(f"  test accuracy (AE on): {acc_g:.2f}%")

    print("\n=== GRADIENT + SPIKING neuron-AE on MNIST ===")
    s = SpikingNAN_MNIST(in_dim, hidden=128, num_classes=10, T=15)
    acc_s = train_mnist(s, tr, te, epochs=2, local_weight=1.0, device=device)
    print(f"  test accuracy (AE on): {acc_s:.2f}%")

    # ---- the sweep that ties back to the NK Figure-6 study -------------------
    # Runs increased to 20; T-Test implemented inside the function.
    resolution_difficulty_sweep(model_kind="gradient", runs=20, epochs=2, device=device)

    # Uncommented the spiking sweep!
    resolution_difficulty_sweep(model_kind="spiking", runs=20, epochs=2, device=device)

device: cuda


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.66MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 134kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.26MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.7MB/s]



=== GRADIENT-ONLY neuron-AE on MNIST ===
  test accuracy (AE on): 90.84%

=== GRADIENT + SPIKING neuron-AE on MNIST ===
  test accuracy (AE on): 92.42%

=== GRADIENT sweep: resolution x difficulty ===
 res  classes  AE-off acc  AE-on acc   delta  p-value
   7        2       99.48      99.42   -0.06   0.0000
   7       10       86.60      82.64   -3.96   0.0000
  14        2       99.85      99.78   -0.07   0.0000
  14       10       91.64      90.70   -0.93   0.0000
  28        2       99.95      99.95   -0.00   0.7481
  28       10       94.26      93.04   -1.21   0.0000

Interpretation guide: a POSITIVE delta that grows with resolution 
indicates the neuron-autoencoder is beneficial at larger N scales.

=== SPIKING sweep: resolution x difficulty ===
 res  classes  AE-off acc  AE-on acc   delta  p-value
   7        2       99.84      99.77   -0.07   0.0000
   7       10       91.09      89.56   -1.52   0.0000
  14        2       99.94      99.92   -0.01   0.0553
  14       10       9

In [ ]:
"""
================================================================================
stage6_schedules_and_collapse.py
--------------------------------------------------------------------------------
Directly addresses the supervisor's email. Two additions on top of stage5:

  (A) SEQUENTIAL training schedule. The email: "In the paper they are
      interleaved, while you could train them sequentially (first the
      autoencoders, and then the task learning)." We implement BOTH schedules
      behind one switch so they can be compared head to head:
          - "interleaved" : coin-flip task/AE per batch (the paper's style)
          - "sequential"  : Phase 1 trains ONLY the autoencoders to convergence,
                            then Phase 2 freezes them and trains ONLY the task.

  (B) NEURON-COLLAPSE DIAGNOSTIC. The email: "all autoencoders in the hidden
      layer learn the same set of characteristics, which would not be good...
      but it is not clear whether this happens." So we MEASURE it, three ways:
          1. pairwise cosine similarity of the per-neuron decoder weight vectors
          2. pairwise correlation of the neurons' activations over a data batch
          3. effective rank of the activation matrix (how many neurons are
             doing genuinely independent work)
      A collapsed layer => decoders near-identical, activations highly
      correlated, effective rank ~1. A healthy layer => low similarity, rank ~H.

Reuses NeuronAELayer from the earlier stages (inlined here so the file is
standalone; import it in your repo instead).

Author scaffolding: Jamal Idrissou
================================================================================
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np


# ==============================================================================
# REUSED: the neuron-AE layer (unchanged from stage3_4 / stage5)
# ==============================================================================
class NeuronAELayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.encoder = nn.Linear(in_features, out_features)
        self.dec_w = nn.Parameter(torch.empty(out_features, in_features))
        self.dec_b = nn.Parameter(torch.zeros(out_features, in_features))
        nn.init.kaiming_uniform_(self.dec_w, a=5 ** 0.5)

    def forward(self, x):
        return torch.sigmoid(self.encoder(x))

    def local_loss(self, x):
        a = torch.sigmoid(self.encoder(x))
        x_hat = a.unsqueeze(-1) * self.dec_w.unsqueeze(0) + self.dec_b.unsqueeze(0)
        return F.mse_loss(x_hat, x.unsqueeze(1).expand_as(x_hat))


class NAN_Classifier(nn.Module):
    """Neuron-AE hidden layer + linear classification head."""
    def __init__(self, in_dim, hidden=128, num_classes=10):
        super().__init__()
        self.hidden = NeuronAELayer(in_dim, hidden)
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, x, want_local=False):
        local = self.hidden.local_loss(x) if want_local else None
        logits = self.head(self.hidden(x))
        return (logits, local) if want_local else logits


# ==============================================================================
# (A) TWO TRAINING SCHEDULES behind one switch
# ==============================================================================
def train_interleaved(model, loader, epochs, local_weight, device):
    """Paper-style: each batch is randomly a task OR an autoencoding cycle."""
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if local_weight == 0 or torch.rand(1).item() < 0.5:        # task
                opt.zero_grad(); F.cross_entropy(model(x), y).backward(); opt.step()
            else:                                                       # AE
                opt.zero_grad(); (local_weight * model(x, True)[1]).backward(); opt.step()
    return model


def train_sequential(model, loader, ae_epochs, task_epochs, device):
    """
    Supervisor's suggested variant: FIRST train the autoencoders alone, THEN
    freeze them and train the task head (plus, optionally, leave the encoder
    trainable — here we FREEZE the encoder after Phase 1 so the task head must
    work with the representation the autoencoding produced; this is the cleaner
    test of 'is the AE-shaped representation good for the task?').
    """
    # ---- Phase 1: autoencoders only ----------------------------------------
    # Only encoder + decoder params get gradients; the head sits idle.
    ae_params = list(model.hidden.parameters())
    opt_ae = torch.optim.Adam(ae_params, lr=1e-3)
    for _ in range(ae_epochs):
        for x, _y in loader:
            x = x.to(device)
            opt_ae.zero_grad()
            model.hidden.local_loss(x).backward()
            opt_ae.step()

    # ---- freeze the AE-trained hidden layer --------------------------------
    for p in model.hidden.parameters():
        p.requires_grad_(False)

    # ---- Phase 2: task head only, on the frozen representation -------------
    opt_task = torch.optim.Adam(model.head.parameters(), lr=1e-3)
    for _ in range(task_epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt_task.zero_grad()
            F.cross_entropy(model(x), y).backward()
            opt_task.step()
    return model


@torch.no_grad()
def accuracy(model, loader, device):
    model.eval(); correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item(); total += y.size(0)
    return 100.0 * correct / total


# ==============================================================================
# (B) NEURON-COLLAPSE DIAGNOSTICS  — "do all neurons learn the same thing?"
# ==============================================================================
@torch.no_grad()
def decoder_similarity(layer):
    """
    Mean absolute pairwise cosine similarity of the per-neuron DECODER weight
    vectors. Each neuron's decoder (dec_w[j]) is 'what it reconstructs'. If all
    neurons learned the same characteristic these vectors point the same way and
    this -> 1.0. Independent neurons -> near 0.
    """
    W = layer.dec_w                                  # (H, in)
    Wn = F.normalize(W, dim=1)                        # unit vectors per neuron
    sim = Wn @ Wn.t()                                 # (H, H) cosine sims
    H = sim.size(0)
    off_diag = sim[~torch.eye(H, dtype=torch.bool, device=W.device)]
    return off_diag.abs().mean().item()


@torch.no_grad()
def activation_redundancy(layer, x):
    """
    Two views of how redundant the neurons' OUTPUTS are on real data:
      - mean |pairwise correlation| of activations across the batch (1 = clones)
      - effective rank of the activation matrix via the entropy of its singular
        value spectrum: exp(H(p)) where p = normalised singular values. Ranges
        from 1 (all neurons collapsed onto one direction) to H (fully diverse).
    """
    a = layer(x)                                      # (B, H) activations
    a = a - a.mean(0, keepdim=True)                   # centre per neuron
    # pairwise correlation
    std = a.std(0, keepdim=True) + 1e-8
    z = a / std
    corr = (z.t() @ z) / z.size(0)                    # (H, H)
    H = corr.size(0)
    off = corr[~torch.eye(H, dtype=torch.bool, device=x.device)]
    mean_abs_corr = off.abs().mean().item()
    # effective rank
    sv = torch.linalg.svdvals(a)                      # singular values
    p = sv / sv.sum()
    entropy = -(p * (p + 1e-12).log()).sum()
    eff_rank = torch.exp(entropy).item()
    return mean_abs_corr, eff_rank, H


def collapse_report(model, x, tag=""):
    """Print the three collapse metrics for a model's hidden layer."""
    sim = decoder_similarity(model.hidden)
    corr, rank, H = activation_redundancy(model.hidden, x)
    print(f"  [{tag}] decoder cos-sim={sim:.3f} | act |corr|={corr:.3f} | "
          f"eff.rank={rank:.1f}/{H}")
    return sim, corr, rank


# ==============================================================================
# Anti-collapse mechanisms (creative direction #1, implemented as a switch)
# ==============================================================================
def diversity_penalty(layer):
    """
    Optional regulariser that PUSHES neurons apart: penalise the mean squared
    off-diagonal cosine similarity of decoder weights. Add lambda*this to the
    autoencoding-cycle loss to actively discourage collapse. Returns a scalar.
    """
    Wn = F.normalize(layer.dec_w, dim=1)
    sim = Wn @ Wn.t()
    H = sim.size(0)
    off = sim[~torch.eye(H, dtype=torch.bool, device=layer.dec_w.device)]
    return (off ** 2).mean()


# ==============================================================================
# Synthetic-or-real MNIST loader (same fallback approach as stage5)
# ==============================================================================
def get_loaders(resolution=14, num_classes=10, batch_size=128, root="./data"):
    from torchvision import datasets, transforms
    tf = transforms.Compose([transforms.Resize((resolution, resolution)),
                             transforms.ToTensor()])
    def prep(ds):
        X, y = [], []
        for img, lab in ds:
            if lab < num_classes:
                X.append(img.view(-1)); y.append(lab)
        return TensorDataset(torch.stack(X), torch.tensor(y))
    try:
        tr = prep(datasets.MNIST(root, train=True, download=True, transform=tf))
        te = prep(datasets.MNIST(root, train=False, download=True, transform=tf))
    except Exception as e:
        print(f"[get_loaders] real MNIST unavailable ({type(e).__name__}); synthetic stand-in.")
        d = resolution * resolution
        def synth(n):
            y = torch.randint(0, num_classes, (n,))
            X = (torch.rand(n, d) * 0.1 + (y.float().unsqueeze(1) / num_classes)).clamp(0, 1)
            return TensorDataset(X, y)
        tr, te = synth(2000), synth(1000)
    return (DataLoader(tr, batch_size=batch_size, shuffle=True),
            DataLoader(te, batch_size=batch_size, shuffle=False),
            resolution * resolution)


# ==============================================================================
# RUNNER: interleaved vs sequential vs baseline, each with a collapse report
# ==============================================================================
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0)
    tr, te, in_dim = get_loaders(resolution=14, num_classes=10)
    xb = next(iter(te))[0].to(device)        # a batch for the diagnostics

    print("=== BASELINE (traditional model, no autoencoding) ===")
    base = NAN_Classifier(in_dim).to(device)
    train_interleaved(base, tr, epochs=3, local_weight=0.0, device=device)
    print(f"  test acc: {accuracy(base, te, device):.2f}%")

    print("\n=== INTERLEAVED neuron-AE (paper style) ===")
    inter = NAN_Classifier(in_dim).to(device)
    train_interleaved(inter, tr, epochs=3, local_weight=1.0, device=device)
    print(f"  test acc: {accuracy(inter, te, device):.2f}%")
    collapse_report(inter, xb, tag="interleaved")

    print("\n=== SEQUENTIAL neuron-AE (AEs first, then frozen task) ===")
    seq = NAN_Classifier(in_dim).to(device)
    train_sequential(seq, tr, ae_epochs=3, task_epochs=3, device=device)
    print(f"  test acc: {accuracy(seq, te, device):.2f}%")
    collapse_report(seq, xb, tag="sequential")

    print("\nNote: compare the collapse metrics across schedules. The "
          "supervisor's open question is whether the neurons diversify; these "
          "three numbers answer it empirically. If collapse appears, switch on "
          "diversity_penalty() in the AE cycle and re-measure.")

=== BASELINE (traditional model, no autoencoding) ===
  test acc: 91.70%

=== INTERLEAVED neuron-AE (paper style) ===
  test acc: 90.71%
  [interleaved] decoder cos-sim=0.533 | act |corr|=0.347 | eff.rank=30.0/128

=== SEQUENTIAL neuron-AE (AEs first, then frozen task) ===
  test acc: 34.88%
  [sequential] decoder cos-sim=0.994 | act |corr|=0.990 | eff.rank=4.5/128

Note: compare the collapse metrics across schedules. The supervisor's open question is whether the neurons diversify; these three numbers answer it empirically. If collapse appears, switch on diversity_penalty() in the AE cycle and re-measure.


In [ ]:
"""
================================================================================
stage6_schedules_and_collapse.py
--------------------------------------------------------------------------------
Directly addresses the supervisor's email. Two additions on top of stage5:

  (A) SEQUENTIAL training schedule. The email: "In the paper they are
      interleaved, while you could train them sequentially (first the
      autoencoders, and then the task learning)." We implement BOTH schedules
      behind one switch so they can be compared head to head:
          - "interleaved" : coin-flip task/AE per batch (the paper's style)
          - "sequential"  : Phase 1 trains ONLY the autoencoders to convergence,
                            then Phase 2 freezes them and trains ONLY the task.

  (B) NEURON-COLLAPSE DIAGNOSTIC. The email: "all autoencoders in the hidden
      layer learn the same set of characteristics, which would not be good...
      but it is not clear whether this happens." So we MEASURE it, three ways:
          1. pairwise cosine similarity of the per-neuron decoder weight vectors
          2. pairwise correlation of the neurons' activations over a data batch
          3. effective rank of the activation matrix (how many neurons are
             doing genuinely independent work)
      A collapsed layer => decoders near-identical, activations highly
      correlated, effective rank ~1. A healthy layer => low similarity, rank ~H.

Reuses NeuronAELayer from the earlier stages (inlined here so the file is
standalone; import it in your repo instead).

Author scaffolding: Jamal Idrissou
================================================================================
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np


# ==============================================================================
# REUSED: the neuron-AE layer (unchanged from stage3_4 / stage5)
# ==============================================================================
class NeuronAELayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.encoder = nn.Linear(in_features, out_features)
        self.dec_w = nn.Parameter(torch.empty(out_features, in_features))
        self.dec_b = nn.Parameter(torch.zeros(out_features, in_features))
        nn.init.kaiming_uniform_(self.dec_w, a=5 ** 0.5)

    def forward(self, x):
        return torch.sigmoid(self.encoder(x))

    def local_loss(self, x):
        a = torch.sigmoid(self.encoder(x))
        x_hat = a.unsqueeze(-1) * self.dec_w.unsqueeze(0) + self.dec_b.unsqueeze(0)
        return F.mse_loss(x_hat, x.unsqueeze(1).expand_as(x_hat))


class NAN_Classifier(nn.Module):
    """Neuron-AE hidden layer + linear classification head."""
    def __init__(self, in_dim, hidden=128, num_classes=10):
        super().__init__()
        self.hidden = NeuronAELayer(in_dim, hidden)
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, x, want_local=False):
        local = self.hidden.local_loss(x) if want_local else None
        logits = self.head(self.hidden(x))
        return (logits, local) if want_local else logits


# ==============================================================================
# Anti-collapse mechanisms (creative direction #1)
# ==============================================================================
def diversity_penalty(layer):
    """
    Optional regulariser that PUSHES neurons apart: penalise the mean squared
    off-diagonal cosine similarity of decoder weights. Add lambda*this to the
    autoencoding-cycle loss to actively discourage collapse. Returns a scalar.
    """
    Wn = F.normalize(layer.dec_w, dim=1)
    sim = Wn @ Wn.t()
    H = sim.size(0)
    off = sim[~torch.eye(H, dtype=torch.bool, device=layer.dec_w.device)]
    return (off ** 2).mean()


# ==============================================================================
# (A) TWO TRAINING SCHEDULES behind one switch
# ==============================================================================
def train_interleaved(model, loader, epochs, local_weight, device, div_weight=0.0):
    """Paper-style: each batch is randomly a task OR an autoencoding cycle."""
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if local_weight == 0 or torch.rand(1).item() < 0.5:        # task
                opt.zero_grad()
                F.cross_entropy(model(x), y).backward()
                opt.step()
            else:                                                      # AE
                opt.zero_grad()
                _, local_loss = model(x, want_local=True)

                # --- APPLY DIVERSITY PENALTY HERE ---
                div_loss = diversity_penalty(model.hidden) if div_weight > 0 else 0.0
                total_ae_loss = (local_weight * local_loss) + (div_weight * div_loss)

                total_ae_loss.backward()
                opt.step()
    return model


def train_sequential(model, loader, ae_epochs, task_epochs, device, div_weight=0.0):
    """
    Supervisor's suggested variant: FIRST train the autoencoders alone, THEN
    freeze them and train the task head.
    """
    # ---- Phase 1: autoencoders only ----------------------------------------
    ae_params = list(model.hidden.parameters())
    opt_ae = torch.optim.Adam(ae_params, lr=1e-3)
    for _ in range(ae_epochs):
        for x, _y in loader:
            x = x.to(device)
            opt_ae.zero_grad()

            local_loss = model.hidden.local_loss(x)

            # --- APPLY DIVERSITY PENALTY HERE ---
            div_loss = diversity_penalty(model.hidden) if div_weight > 0 else 0.0
            total_ae_loss = local_loss + (div_weight * div_loss)

            total_ae_loss.backward()
            opt_ae.step()

    # ---- freeze the AE-trained hidden layer --------------------------------
    for p in model.hidden.parameters():
        p.requires_grad_(False)

    # ---- Phase 2: task head only, on the frozen representation -------------
    opt_task = torch.optim.Adam(model.head.parameters(), lr=1e-3)
    for _ in range(task_epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt_task.zero_grad()
            F.cross_entropy(model(x), y).backward()
            opt_task.step()
    return model


@torch.no_grad()
def accuracy(model, loader, device):
    model.eval(); correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item(); total += y.size(0)
    return 100.0 * correct / total


# ==============================================================================
# (B) NEURON-COLLAPSE DIAGNOSTICS  — "do all neurons learn the same thing?"
# ==============================================================================
@torch.no_grad()
def decoder_similarity(layer):
    W = layer.dec_w                                  # (H, in)
    Wn = F.normalize(W, dim=1)                        # unit vectors per neuron
    sim = Wn @ Wn.t()                                 # (H, H) cosine sims
    H = sim.size(0)
    off_diag = sim[~torch.eye(H, dtype=torch.bool, device=W.device)]
    return off_diag.abs().mean().item()


@torch.no_grad()
def activation_redundancy(layer, x):
    a = layer(x)                                      # (B, H) activations
    a = a - a.mean(0, keepdim=True)                   # centre per neuron
    # pairwise correlation
    std = a.std(0, keepdim=True) + 1e-8
    z = a / std
    corr = (z.t() @ z) / z.size(0)                    # (H, H)
    H = corr.size(0)
    off = corr[~torch.eye(H, dtype=torch.bool, device=x.device)]
    mean_abs_corr = off.abs().mean().item()
    # effective rank
    sv = torch.linalg.svdvals(a)                      # singular values
    p = sv / sv.sum()
    entropy = -(p * (p + 1e-12).log()).sum()
    eff_rank = torch.exp(entropy).item()
    return mean_abs_corr, eff_rank, H


def collapse_report(model, x, tag=""):
    sim = decoder_similarity(model.hidden)
    corr, rank, H = activation_redundancy(model.hidden, x)
    print(f"  [{tag}] decoder cos-sim={sim:.3f} | act |corr|={corr:.3f} | "
          f"eff.rank={rank:.1f}/{H}")
    return sim, corr, rank


# ==============================================================================
# Synthetic-or-real MNIST loader (same fallback approach as stage5)
# ==============================================================================
def get_loaders(resolution=14, num_classes=10, batch_size=128, root="./data"):
    from torchvision import datasets, transforms
    tf = transforms.Compose([transforms.Resize((resolution, resolution)),
                             transforms.ToTensor()])
    def prep(ds):
        X, y = [], []
        for img, lab in ds:
            if lab < num_classes:
                X.append(img.view(-1)); y.append(lab)
        return TensorDataset(torch.stack(X), torch.tensor(y))
    try:
        tr = prep(datasets.MNIST(root, train=True, download=True, transform=tf))
        te = prep(datasets.MNIST(root, train=False, download=True, transform=tf))
    except Exception as e:
        print(f"[get_loaders] real MNIST unavailable ({type(e).__name__}); synthetic stand-in.")
        d = resolution * resolution
        def synth(n):
            y = torch.randint(0, num_classes, (n,))
            X = (torch.rand(n, d) * 0.1 + (y.float().unsqueeze(1) / num_classes)).clamp(0, 1)
            return TensorDataset(X, y)
        tr, te = synth(2000), synth(1000)
    return (DataLoader(tr, batch_size=batch_size, shuffle=True),
            DataLoader(te, batch_size=batch_size, shuffle=False),
            resolution * resolution)


# ==============================================================================
# RUNNER: interleaved vs sequential vs baseline, each with a collapse report
# ==============================================================================
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0)
    tr, te, in_dim = get_loaders(resolution=14, num_classes=10)
    xb = next(iter(te))[0].to(device)        # a batch for the diagnostics

    print("=== BASELINE (traditional model, no autoencoding) ===")
    base = NAN_Classifier(in_dim).to(device)
    train_interleaved(base, tr, epochs=3, local_weight=0.0, device=device)
    print(f"  test acc: {accuracy(base, te, device):.2f}%")

    print("\n=== INTERLEAVED neuron-AE (paper style) ===")
    inter = NAN_Classifier(in_dim).to(device)
    train_interleaved(inter, tr, epochs=3, local_weight=1.0, device=device)
    print(f"  test acc: {accuracy(inter, te, device):.2f}%")
    collapse_report(inter, xb, tag="interleaved")

    print("\n=== SEQUENTIAL neuron-AE (AEs first, then frozen task) ===")
    seq = NAN_Classifier(in_dim).to(device)
    train_sequential(seq, tr, ae_epochs=3, task_epochs=3, device=device)
    print(f"  test acc: {accuracy(seq, te, device):.2f}%")
    collapse_report(seq, xb, tag="sequential")

    print("\n=== SEQUENTIAL neuron-AE (WITH DIVERSITY PENALTY) ===")
    seq_fixed = NAN_Classifier(in_dim).to(device)
    # Note the div_weight=5.0 here to forcefully push the neurons apart
    train_sequential(seq_fixed, tr, ae_epochs=3, task_epochs=3, device=device, div_weight=5.0)
    print(f"  test acc: {accuracy(seq_fixed, te, device):.2f}%")
    collapse_report(seq_fixed, xb, tag="sequential_fixed")

    print("\nNote: Compare 'sequential' against 'sequential_fixed'.")
    print("The diversity penalty should dramatically increase the effective rank ")
    print("and rescue the test accuracy.")

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.04MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 134kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.26MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.27MB/s]


=== BASELINE (traditional model, no autoencoding) ===
  test acc: 91.70%

=== INTERLEAVED neuron-AE (paper style) ===
  test acc: 90.71%
  [interleaved] decoder cos-sim=0.533 | act |corr|=0.347 | eff.rank=30.0/128

=== SEQUENTIAL neuron-AE (AEs first, then frozen task) ===
  test acc: 34.88%
  [sequential] decoder cos-sim=0.994 | act |corr|=0.990 | eff.rank=4.5/128

=== SEQUENTIAL neuron-AE (WITH DIVERSITY PENALTY) ===
  test acc: 86.30%
  [sequential_fixed] decoder cos-sim=0.001 | act |corr|=0.241 | eff.rank=42.6/128

Note: Compare 'sequential' against 'sequential_fixed'.
The diversity penalty should dramatically increase the effective rank 
and rescue the test accuracy.


In [2]:
"""
================================================================================
stage7_experiment_harness.py
--------------------------------------------------------------------------------
The experiment harness that turns the working code into presentable results.

It adds, on top of stage6:
  1. INSTRUMENTED training: logs test acc, task loss, recon loss, and all three
     collapse metrics at EVERY epoch (not just the end), so you get curves.
  2. MULTI-SEED runs with mean +/- standard-error bands on every curve.
  3. Two SWEEPS: div_weight (anti-collapse strength) and local_weight
     (task-vs-reconstruction trade-off), each producing accuracy + effective
     rank vs the swept variable.
  4. Sequential schedule VARIANTS: freeze vs fine-tune the encoder in Phase 2,
     crossed with/without the diversity penalty (the 2x2 discussed).
  5. Two PENALTY FORMULATIONS: decoder-weight decorrelation (original) and
     activation decorrelation (Barlow-Twins style), selectable by name.
  6. RESULTS SAVED to disk (JSON) so you can re-plot without re-training, and
     PLOTS written to PNG.

Run modes (set MODE at the bottom):
  "demo"        - tiny, fast, just proves the harness runs
  "curves"      - per-epoch curves, interleaved vs sequential vs fixed
  "div_sweep"   - accuracy + eff-rank vs diversity weight
  "local_sweep" - accuracy + eff-rank vs local (reconstruction) weight
  "seq_variants"- the freeze/finetune x penalty 2x2

Author scaffolding: Jamal Idrissou
================================================================================
"""

import json, os, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

OUT = "experiments"
os.makedirs(OUT, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ==============================================================================
# Model (unchanged neuron-AE layer + classifier)
# ==============================================================================
class NeuronAELayer(nn.Module):
    """NAN hidden layer: every neuron is its own autoencoder.

    Each neuron encodes the input vector down to a single scalar (its activation)
    and decodes that scalar back to a full reconstruction of the input. The
    forward pass returns the activations (passed to the next layer); local_loss
    returns how badly each neuron reconstructs its input, averaged.
    """
    def __init__(self, in_features, out_features):
        super().__init__()
        self.encoder = nn.Linear(in_features, out_features)
        # One decoder PER neuron: row j maps neuron j's scalar -> full input.
        # Shapes are (num_neurons, in_features).
        self.decoder_weight = nn.Parameter(torch.empty(out_features, in_features))
        self.decoder_bias = nn.Parameter(torch.zeros(out_features, in_features))
        nn.init.kaiming_uniform_(self.decoder_weight, a=5 ** 0.5)

    def forward(self, x):
        # the scalar "code" each neuron emits; this is the normal layer output
        return torch.sigmoid(self.encoder(x))

    def local_loss(self, x):
        code = torch.sigmoid(self.encoder(x))          # (batch, num_neurons)
        # reconstruct the input from EACH neuron's scalar, independently:
        #   reconstruction[b, j, :] = code[b,j] * decoder_weight[j] + decoder_bias[j]
        reconstruction = (code.unsqueeze(-1) * self.decoder_weight.unsqueeze(0)
                          + self.decoder_bias.unsqueeze(0))   # (batch, neurons, in)
        target = x.unsqueeze(1).expand_as(reconstruction)     # broadcast input
        return F.mse_loss(reconstruction, target)


# ------------------------------------------------------------------------------
# SPIKING neuron-AE layer (directive compliance). Same INTERFACE as
# NeuronAELayer so the harness, penalties and diagnostics work unchanged:
#   - exposes .encoder (the Linear), .dec_w, .dec_b   (used by diagnostics/penalties)
#   - forward(x) -> per-neuron scalar code (here the firing RATE)
#   - local_loss(x) -> per-neuron reconstruction MSE from that rate
# The only conceptual change vs the static layer: each neuron's scalar "code" is
# its mean firing rate over T leaky-integrate-and-fire timesteps, and the spike
# is made differentiable with a surrogate gradient.
# ------------------------------------------------------------------------------
class _SurrogateSpike(torch.autograd.Function):
    """A spike (hard threshold) with a fake-but-usable gradient.

    A real spike is a step function, whose derivative is zero everywhere and so
    blocks gradient descent. The standard fix: emit the true 0/1 spike on the
    forward pass, but on the backward pass pretend the step was a smooth curve
    so a gradient still flows. This is the 'surrogate gradient' trick.
    """
    @staticmethod
    def forward(ctx, membrane_minus_threshold):
        ctx.save_for_backward(membrane_minus_threshold)
        return (membrane_minus_threshold >= 0).float()      # real spike: 1 or 0

    @staticmethod
    def backward(ctx, grad_output):
        (shifted_membrane,) = ctx.saved_tensors
        # derivative of a steep smooth sigmoid, peaked at the threshold
        surrogate_grad = 1.0 / (10.0 * torch.abs(shifted_membrane) + 1.0) ** 2
        return grad_output * surrogate_grad


spike = _SurrogateSpike.apply


class SpikingNeuronAELayer(nn.Module):
    """Spiking version of NeuronAELayer. Same interface, but each neuron's
    scalar 'code' is its FIRING RATE over `n_timesteps` leaky-integrate-and-fire
    steps, rather than a single static activation."""
    def __init__(self, in_features, out_features,
                 n_timesteps=20, leak=0.9, threshold=1.0):
        super().__init__()
        self.num_neurons = out_features
        self.n_timesteps = n_timesteps
        self.leak = leak                                # membrane decay per step
        self.threshold = threshold                      # voltage needed to fire
        self.encoder = nn.Linear(in_features, out_features)
        self.decoder_weight = nn.Parameter(torch.empty(out_features, in_features))
        self.decoder_bias = nn.Parameter(torch.zeros(out_features, in_features))
        nn.init.kaiming_uniform_(self.decoder_weight, a=5 ** 0.5)

    def firing_rate(self, x):
        """Simulate the neurons for n_timesteps; return mean spikes per neuron."""
        batch_size = x.size(0)
        input_current = self.encoder(x)                 # constant drive each step
        membrane = torch.zeros(batch_size, self.num_neurons, device=x.device)
        spike_count = torch.zeros(batch_size, self.num_neurons, device=x.device)
        for _ in range(self.n_timesteps):
            membrane = self.leak * membrane + input_current        # integrate
            fired = spike(membrane - self.threshold)               # 1 where fired
            membrane = membrane * (1.0 - fired)                    # reset on fire
            spike_count = spike_count + fired
        return spike_count / self.n_timesteps           # rate in [0, 1]

    def forward(self, x):
        return self.firing_rate(x)                      # the neuron's scalar code

    def local_loss(self, x):
        rate = self.firing_rate(x)                      # (batch, num_neurons)
        reconstruction = (rate.unsqueeze(-1) * self.decoder_weight.unsqueeze(0)
                          + self.decoder_bias.unsqueeze(0))
        target = x.unsqueeze(1).expand_as(reconstruction)
        return F.mse_loss(reconstruction, target)


# map a string to the hidden-layer class so experiments can switch architecture
# ------------------------------------------------------------------------------
# The paper's OTHER two architectures, so all three of Bull's variants are
# selectable. All share the same interface (.encoder, .dec_w, .dec_b, forward,
# local_loss) so the harness/diagnostics/penalties work on them unchanged.
#
#   NN  : plain MLP hidden layer. NO autoencoder at all -> local_loss == 0, so
#         with the alternating trainer it only ever runs task cycles. This is
#         the honest "traditional model" baseline (distinct from a NAN with
#         local_weight=0, which still carries unused decoder parameters).
#   ANN : ONE shared decoder layer (H -> input) hanging off the whole hidden
#         layer — the standard "one decoder layer" autoencoder in the paper.
#         Contrast with NAN, where each neuron has its OWN 1->input decoder.
# ------------------------------------------------------------------------------
class PlainNNLayer(nn.Module):
    """Plain hidden layer, no autoencoding (Bull's 'NN'): the traditional baseline."""
    def __init__(self, in_features, out_features):
        super().__init__()
        self.encoder = nn.Linear(in_features, out_features)
        # Unused placeholder decoder params, frozen, so the diagnostics that look
        # for .decoder_weight still find an attribute. Never trained, because this
        # layer's reconstruction loss is identically zero.
        self.decoder_weight = nn.Parameter(torch.zeros(out_features, in_features),
                                           requires_grad=False)
        self.decoder_bias = nn.Parameter(torch.zeros(out_features, in_features),
                                         requires_grad=False)

    def forward(self, x):
        return torch.sigmoid(self.encoder(x))

    def local_loss(self, x):
        # no autoencoder -> a true zero with no gradient path; the trainer detects
        # this and simply skips the autoencoding cycle for a plain NN.
        return torch.zeros((), device=x.device)


class LayerAELayer(nn.Module):
    """Single SHARED decoder over the whole hidden layer (Bull's 'ANN')."""
    def __init__(self, in_features, out_features):
        super().__init__()
        self.encoder = nn.Linear(in_features, out_features)
        self.decoder = nn.Linear(out_features, in_features)   # hidden -> input
        # Expose the decoder weights under the same name the NAN layers use, so
        # the diagnostics read something. Note: here a row corresponds to an
        # INPUT dimension, not a neuron, so the decoder-similarity metric isn't
        # meaningful for ANN — the activation metrics are the ones to trust.
        self.decoder_weight = self.decoder.weight
        self.decoder_bias = self.decoder.bias

    def forward(self, x):
        return torch.sigmoid(self.encoder(x))

    def local_loss(self, x):
        hidden_code = torch.sigmoid(self.encoder(x))   # (batch, num_neurons)
        reconstruction = self.decoder(hidden_code)     # (batch, in_features)
        return F.mse_loss(reconstruction, x)


# selectable architectures. "static"/"spiking" are NANs (per-neuron decoders);
# "ann" is the shared-decoder autoencoder; "nn" is the plain baseline.
ARCHS = {
    "static":  NeuronAELayer,          # NAN, static neurons (paper's proposal)
    "spiking": SpikingNeuronAELayer,   # NAN, spiking neurons (your extension)
    "ann":     LayerAELayer,           # ANN, one shared decoder layer
    "nn":      PlainNNLayer,           # NN, no autoencoder (traditional baseline)
}


class HiddenLayerClassifier(nn.Module):
    """A one-hidden-layer classifier where the hidden layer can be any of the
    four architectures (NN / ANN / NAN-static / NAN-spiking), chosen by `arch`."""
    def __init__(self, in_dim, hidden=128, num_classes=10, arch="static"):
        super().__init__()
        self.hidden = ARCHS[arch](in_dim, hidden)   # the chosen hidden layer type
        self.head = nn.Linear(hidden, num_classes)  # linear readout to class logits

    def forward(self, x, want_local=False):
        recon_loss = self.hidden.local_loss(x) if want_local else None
        logits = self.head(self.hidden(x))
        return (logits, recon_loss) if want_local else logits


# Backwards-compatible alias (older call sites used this name).
NAN_Classifier = HiddenLayerClassifier


# ==============================================================================
# Diversity penalties — TWO formulations, selectable by name
# ==============================================================================
def penalty_decoder(layer, x=None):
    """Push the per-neuron DECODER weight vectors apart (decorrelate them).
    Acts on parameters: discourages neurons from reconstructing the same thing."""
    unit_vectors = F.normalize(layer.decoder_weight, dim=1)   # one unit vec/neuron
    similarity = unit_vectors @ unit_vectors.t()              # (neurons, neurons)
    num_neurons = similarity.size(0)
    off_diagonal = similarity[~torch.eye(num_neurons, dtype=torch.bool,
                                         device=layer.decoder_weight.device)]
    return (off_diagonal ** 2).mean()                         # 0 when orthogonal


def penalty_activation(layer, x):
    """Barlow-Twins style: decorrelate the ACTIVATIONS over a batch.
    Acts on the representation (what neurons actually output). Needs the input x."""
    activations = layer(x)                                    # (batch, neurons)
    activations = ((activations - activations.mean(0, keepdim=True))
                   / (activations.std(0, keepdim=True) + 1e-8))
    correlation = (activations.t() @ activations) / activations.size(0)
    num_neurons = correlation.size(0)
    off_diagonal = correlation[~torch.eye(num_neurons, dtype=torch.bool,
                                          device=x.device)]
    return (off_diagonal ** 2).mean()                         # push off-diag to 0


PENALTIES = {"decoder": penalty_decoder, "activation": penalty_activation}


# ==============================================================================
# Diagnostics (same three metrics as stage6)
# ==============================================================================
@torch.no_grad()
def metrics(model, loader, probe_batch):
    """Compute test accuracy plus three 'neuron collapse' diagnostics.

    probe_batch: a fixed batch of inputs used to measure how diverse the neurons
    are. The three collapse metrics answer the supervisor's question 'do all the
    neurons end up learning the same thing?':
        decoder_similarity : how alike the neurons' decoder weights are (NAN only)
        activation_corr    : how correlated the neurons' outputs are
        effective_rank     : how many neurons do genuinely independent work
    """
    model.eval()

    # ---- test accuracy ----
    correct = total = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        correct += (model(x).argmax(1) == y).sum().item()
        total += y.size(0)
    accuracy = 100.0 * correct / total

    layer = model.hidden

    # ---- decoder similarity (only meaningful when each neuron has its own
    #      decoder row, i.e. the NAN layers). For NN the decoder is unused, for
    #      ANN it is shared and indexed by input dimension, so we report -1
    #      ('not applicable') in those cases. ----
    num_neurons = layer.encoder.out_features
    decoder_is_per_neuron = (layer.decoder_weight.shape[0] == num_neurons
                             and layer.decoder_weight.abs().sum() > 0)
    if decoder_is_per_neuron:
        unit_vectors = F.normalize(layer.decoder_weight, dim=1)
        similarity = unit_vectors @ unit_vectors.t()
        off_diagonal = similarity[~torch.eye(num_neurons, dtype=torch.bool,
                                             device=probe_batch.device)]
        decoder_similarity = off_diagonal.abs().mean().item()
    else:
        decoder_similarity = -1.0     # not applicable for NN / ANN

    # ---- activation correlation: centre and standardise each neuron's outputs
    #      over the batch, then average the off-diagonal of the correlation
    #      matrix (1 = neurons are clones, 0 = uncorrelated). ----
    activations = layer(probe_batch)
    activations = activations - activations.mean(0, keepdim=True)
    standardised = activations / (activations.std(0, keepdim=True) + 1e-8)
    correlation = (standardised.t() @ standardised) / standardised.size(0)
    n = correlation.size(0)
    activation_corr = correlation[~torch.eye(n, dtype=torch.bool,
                                             device=probe_batch.device)].abs().mean().item()

    # ---- effective rank: how many neurons are effectively independent. We take
    #      the singular values of the activation matrix, turn them into a
    #      probability distribution, and compute exp(entropy). Ranges from 1
    #      (total collapse onto one direction) up to num_neurons (full diversity). ----
    singular_values = torch.linalg.svdvals(activations)
    spectrum = singular_values / singular_values.sum()
    entropy = -(spectrum * (spectrum + 1e-12).log()).sum()
    effective_rank = torch.exp(entropy).item()

    return {"acc": accuracy, "dec_sim": decoder_similarity,
            "act_corr": activation_corr, "eff_rank": effective_rank}


# ==============================================================================
# Instrumented training — logs metrics every epoch
# ==============================================================================
def run_interleaved(model, train_loader, test_loader, probe_batch, epochs,
                    local_weight, div_weight=0.0, penalty="decoder"):
    """Paper-style schedule: each batch is randomly EITHER a task-learning cycle
    OR an autoencoding cycle. Records the metrics once per epoch."""
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    penalty_fn = PENALTIES[penalty]
    history = []
    for _epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            run_task_cycle = local_weight == 0 or torch.rand(1).item() < 0.5
            if run_task_cycle:
                optimizer.zero_grad()
                F.cross_entropy(model(x), y).backward()
                optimizer.step()
            else:
                optimizer.zero_grad()
                _, recon_loss = model(x, want_local=True)
                loss = local_weight * recon_loss
                if div_weight > 0:
                    loss = loss + div_weight * penalty_fn(model.hidden, x)
                # A plain NN has no autoencoder: its reconstruction loss is a
                # constant zero with no gradient path. There is nothing to learn
                # on an autoencoding cycle, so fall back to a task cycle rather
                # than calling backward() on a non-differentiable tensor.
                if loss.requires_grad:
                    loss.backward()
                    optimizer.step()
                else:
                    F.cross_entropy(model(x), y).backward()
                    optimizer.step()
        history.append(metrics(model, test_loader, probe_batch))
    return history


def run_sequential(model, train_loader, test_loader, probe_batch,
                   ae_epochs, task_epochs,
                   div_weight=0.0, penalty="decoder", finetune=False):
    """
    Sequential schedule (the variant the supervisor suggested):
      Phase 1 trains ONLY the autoencoders (optionally with a diversity penalty).
      Phase 2 trains the task. If finetune=True the encoder stays trainable at a
        low learning rate so the task signal can repair leftover collapse; if
        finetune=False the hidden layer is frozen, the strict 'is the AE-shaped
        representation good for the task?' test.
    Metrics are logged every epoch across BOTH phases, so a curve shows the
    Phase-1 -> Phase-2 transition.
    """
    penalty_fn = PENALTIES[penalty]
    history = []

    # ---- Phase 1: train the autoencoders (skipped for a plain NN, which has
    #      no autoencoder to train) ----
    first_batch = next(iter(train_loader))[0].to(DEVICE)
    has_autoencoder = model.hidden.local_loss(first_batch).requires_grad
    ae_optimizer = torch.optim.Adam(model.hidden.parameters(), lr=1e-3)
    for _epoch in range(ae_epochs):
        model.train()
        if has_autoencoder:
            for x, _y in train_loader:
                x = x.to(DEVICE)
                ae_optimizer.zero_grad()
                loss = model.hidden.local_loss(x)
                if div_weight > 0:
                    loss = loss + div_weight * penalty_fn(model.hidden, x)
                loss.backward()
                ae_optimizer.step()
        history.append({**metrics(model, test_loader, probe_batch), "phase": 1})

    # ---- Phase 2: train the task ----
    if finetune:
        # encoder keeps learning, but slowly, while the head learns at full speed
        task_optimizer = torch.optim.Adam([
            {"params": model.head.parameters(), "lr": 1e-3},
            {"params": model.hidden.parameters(), "lr": 1e-4},
        ])
    else:
        for param in model.hidden.parameters():
            param.requires_grad_(False)             # freeze the AE representation
        task_optimizer = torch.optim.Adam(model.head.parameters(), lr=1e-3)

    for _epoch in range(task_epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            task_optimizer.zero_grad()
            F.cross_entropy(model(x), y).backward()
            task_optimizer.step()
        history.append({**metrics(model, test_loader, probe_batch), "phase": 2})
    return history


# ==============================================================================
# Multi-seed wrapper: returns per-epoch mean and standard error for each metric
# ==============================================================================
def multiseed(make_and_run, seeds, in_dim, hidden, num_classes):
    """Run an experiment once per seed and aggregate the per-epoch metrics into
    a mean and standard-error curve.

    make_and_run(seed) -> a list (one entry per epoch) of metric dicts.
    Returns {metric_name: {'mean': [...per epoch...], 'sem': [...]}}.
    """
    runs_per_seed = []
    for seed in seeds:
        torch.manual_seed(seed)
        np.random.seed(seed)
        runs_per_seed.append(make_and_run(seed))

    metric_names = [name for name in runs_per_seed[0][0].keys() if name != "phase"]
    aggregated = {}
    for name in metric_names:
        # shape (num_seeds, num_epochs): this metric's value at each epoch per seed
        values = np.array([[epoch[name] for epoch in run] for run in runs_per_seed])
        aggregated[name] = {
            "mean": values.mean(0).tolist(),
            "sem": (values.std(0) / np.sqrt(len(seeds))).tolist(),
        }
    # carry the phase markers (for the sequential schedule) straight through
    aggregated["phase"] = [epoch.get("phase", 0) for epoch in runs_per_seed[0]]
    return aggregated


# ==============================================================================
# Data
# ==============================================================================
def get_loaders(resolution=14, num_classes=10, batch_size=128, root="./data"):
    from torchvision import datasets, transforms
    tf = transforms.Compose([transforms.Resize((resolution, resolution)),
                             transforms.ToTensor()])
    def prep(ds):
        X, y = [], []
        for img, lab in ds:
            if lab < num_classes:
                X.append(img.view(-1)); y.append(lab)
        return TensorDataset(torch.stack(X), torch.tensor(y))
    try:
        tr = prep(datasets.MNIST(root, train=True, download=True, transform=tf))
        te = prep(datasets.MNIST(root, train=False, download=True, transform=tf))
    except Exception as e:
        print(f"[get_loaders] real MNIST unavailable ({type(e).__name__}); synthetic stand-in.")
        d = resolution * resolution
        def synth(n):
            y = torch.randint(0, num_classes, (n,))
            X = (torch.rand(n, d) * 0.1 + (y.float().unsqueeze(1) / num_classes)).clamp(0, 1)
            return TensorDataset(X, y)
        tr, te = synth(2000), synth(1000)
    return (DataLoader(tr, batch_size=batch_size, shuffle=True),
            DataLoader(te, batch_size=batch_size, shuffle=False),
            resolution * resolution)


# ==============================================================================
# Plotting (saves PNGs; no display needed)
# ==============================================================================
def plot_curves(curve_dict, metric, ylabel, fname, title):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 4.5))
    for label, agg in curve_dict.items():
        m = np.array(agg[metric]["mean"]); s = np.array(agg[metric]["sem"])
        ep = np.arange(1, len(m) + 1)
        plt.plot(ep, m, label=label, linewidth=2)
        plt.fill_between(ep, m - s, m + s, alpha=0.2)
    plt.xlabel("epoch"); plt.ylabel(ylabel); plt.title(title)
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    path = os.path.join(OUT, fname); plt.savefig(path, dpi=130); plt.close()
    print(f"  saved {path}")


def plot_sweep(xs, ys_dict, xlabel, fname, title):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax1 = plt.subplots(figsize=(7, 4.5))
    ax2 = ax1.twinx()
    am, asem = ys_dict["acc_mean"], ys_dict["acc_sem"]
    rm, rsem = ys_dict["rank_mean"], ys_dict["rank_sem"]
    ax1.errorbar(xs, am, yerr=asem, marker="o", color="tab:blue", label="test acc")
    ax2.errorbar(xs, rm, yerr=rsem, marker="s", color="tab:red", label="eff. rank")
    ax1.set_xlabel(xlabel); ax1.set_ylabel("test accuracy (%)", color="tab:blue")
    ax2.set_ylabel("effective rank", color="tab:red")
    ax1.set_title(title); ax1.grid(alpha=0.3); fig.tight_layout()
    path = os.path.join(OUT, fname); plt.savefig(path, dpi=130); plt.close()
    print(f"  saved {path}")


def save_json(obj, fname):
    with open(os.path.join(OUT, fname), "w") as f:
        json.dump(obj, f, indent=2)
    print(f"  saved {os.path.join(OUT, fname)}")


# ==============================================================================
# EXPERIMENTS
# ==============================================================================
def exp_curves(seeds, epochs, res=14, nc=10, hidden=128, arch="static"):
    """Per-epoch curves: interleaved, sequential(frozen), sequential(fixed+pen).
    arch: "static" or "spiking" — selects the hidden-layer neuron model."""
    tr, te, in_dim = get_loaders(resolution=res, num_classes=nc)
    xb = next(iter(te))[0].to(DEVICE)
    print(f"exp_curves [{arch}]: interleaved ...")
    inter = multiseed(lambda s: run_interleaved(
        NAN_Classifier(in_dim, hidden, nc, arch=arch).to(DEVICE), tr, te, xb, epochs, 1.0),
        seeds, in_dim, hidden, nc)
    print(f"exp_curves [{arch}]: sequential (frozen, no penalty) ...")
    seq = multiseed(lambda s: run_sequential(
        NAN_Classifier(in_dim, hidden, nc, arch=arch).to(DEVICE), tr, te, xb,
        epochs // 2, epochs - epochs // 2, div_weight=0.0, finetune=False),
        seeds, in_dim, hidden, nc)
    print(f"exp_curves [{arch}]: sequential (frozen + diversity penalty) ...")
    seqp = multiseed(lambda s: run_sequential(
        NAN_Classifier(in_dim, hidden, nc, arch=arch).to(DEVICE), tr, te, xb,
        epochs // 2, epochs - epochs // 2, div_weight=5.0, finetune=False),
        seeds, in_dim, hidden, nc)
    curves = {"interleaved": inter, "sequential": seq, "sequential+penalty": seqp}
    save_json(curves, f"curves_{arch}.json")
    plot_curves(curves, "acc", "test accuracy (%)", f"curve_acc_{arch}.png",
                f"Test accuracy over training ({arch})")
    plot_curves(curves, "eff_rank", "effective rank", f"curve_rank_{arch}.png",
                f"Neuron diversity (effective rank) over training ({arch})")
    return curves


def exp_sweep(kind, values, seeds, epochs, res=14, nc=10, hidden=128, arch="static"):
    """Sweep div_weight (kind='div') or local_weight (kind='local').
    Records FINAL accuracy and eff-rank (mean+sem over seeds) per value.
    arch: "static" or "spiking"."""
    tr, te, in_dim = get_loaders(resolution=res, num_classes=nc)
    xb = next(iter(te))[0].to(DEVICE)
    acc_m, acc_s, rank_m, rank_s = [], [], [], []
    for v in values:
        print(f"exp_sweep[{kind}/{arch}]: value={v} ...")
        def run(s):
            m = NAN_Classifier(in_dim, hidden, nc, arch=arch).to(DEVICE)
            if kind == "div":   # interleaved, vary diversity penalty
                return run_interleaved(m, tr, te, xb, epochs, 1.0, div_weight=v)
            else:               # vary local (reconstruction) weight, no penalty
                return run_interleaved(m, tr, te, xb, epochs, v, div_weight=0.0)
        agg = multiseed(run, seeds, in_dim, hidden, nc)
        acc_m.append(agg["acc"]["mean"][-1]);  acc_s.append(agg["acc"]["sem"][-1])
        rank_m.append(agg["eff_rank"]["mean"][-1]); rank_s.append(agg["eff_rank"]["sem"][-1])
    res_obj = {"values": values, "acc_mean": acc_m, "acc_sem": acc_s,
               "rank_mean": rank_m, "rank_sem": rank_s}
    save_json(res_obj, f"sweep_{kind}_{arch}.json")
    xlabel = "diversity weight" if kind == "div" else "local (reconstruction) weight"
    plot_sweep(values, res_obj, xlabel, f"sweep_{kind}_{arch}.png",
               f"Final accuracy & neuron diversity vs {xlabel} ({arch})")
    return res_obj


def exp_seq_variants(seeds, epochs, res=14, nc=10, hidden=128, arch="static"):
    """The 2x2: {freeze, finetune} x {no penalty, penalty}. Final metrics.
    arch: "static" or "spiking"."""
    tr, te, in_dim = get_loaders(resolution=res, num_classes=nc)
    xb = next(iter(te))[0].to(DEVICE)
    grid = {}
    for finetune in (False, True):
        for dw in (0.0, 5.0):
            name = f"{'finetune' if finetune else 'freeze'}+{'pen' if dw>0 else 'nopen'}"
            print(f"exp_seq_variants [{arch}]: {name} ...")
            agg = multiseed(lambda s: run_sequential(
                NAN_Classifier(in_dim, hidden, nc, arch=arch).to(DEVICE), tr, te, xb,
                epochs // 2, epochs - epochs // 2, div_weight=dw, finetune=finetune),
                seeds, in_dim, hidden, nc)
            grid[name] = {"acc": agg["acc"]["mean"][-1],
                          "acc_sem": agg["acc"]["sem"][-1],
                          "eff_rank": agg["eff_rank"]["mean"][-1]}
    save_json(grid, f"seq_variants_{arch}.json")
    print(f"\n  2x2 sequential variants ({arch}, final):")
    print(f"  {'variant':20} {'acc':>8} {'+-sem':>7} {'eff_rank':>9}")
    for k, v in grid.items():
        print(f"  {k:20} {v['acc']:>8.2f} {v['acc_sem']:>7.2f} {v['eff_rank']:>9.1f}")
    return grid


# ==============================================================================
if __name__ == "__main__":
    MODE = "demo"   # "demo"|"curves"|"div_sweep"|"local_sweep"|"seq_variants"|"compare_arch"|"compare_all4"
    ARCH = "static" # "static"(NAN) | "spiking"(spiking NAN) | "ann"(shared decoder) | "nn"(plain baseline)
    t0 = time.time()
    print(f"device={DEVICE} mode={MODE} arch={ARCH}")

    if MODE == "demo":
        # tiny everything, run BOTH NAN archs so directive compliance is visible
        exp_curves(seeds=[0, 1], epochs=2, res=7, nc=10, hidden=64, arch="static")
        exp_curves(seeds=[0, 1], epochs=2, res=7, nc=10, hidden=64, arch="spiking")

    elif MODE == "curves":
        exp_curves(seeds=range(5), epochs=10, arch=ARCH)

    elif MODE == "div_sweep":
        exp_sweep("div", [0, 0.1, 0.5, 1, 2, 5, 10, 20], seeds=range(5), epochs=6, arch=ARCH)

    elif MODE == "local_sweep":
        exp_sweep("local", [0, 0.1, 0.25, 0.5, 1, 2, 4], seeds=range(5), epochs=6, arch=ARCH)

    elif MODE == "seq_variants":
        exp_seq_variants(seeds=range(5), epochs=10, arch=ARCH)

    elif MODE == "compare_arch":
        # same experiment for both NAN neuron models: static vs spiking
        for a in ("static", "spiking"):
            exp_seq_variants(seeds=range(5), epochs=10, arch=a)
            exp_sweep("div", [0, 0.5, 1, 2, 5, 10], seeds=range(5), epochs=6, arch=a)

    elif MODE == "compare_all4":
        # the paper's full architecture comparison: NN vs ANN vs NAN(static) vs
        # NAN(spiking), interleaved training, final accuracy + diversity per arch.
        tr, te, in_dim = get_loaders(resolution=14, num_classes=10)
        xb = next(iter(te))[0].to(DEVICE)
        print(f"\n  {'arch':10} {'acc':>8} {'+-sem':>7} {'eff_rank':>9} {'dec_sim':>8}")
        summary = {}
        for a in ("nn", "ann", "static", "spiking"):
            # nn uses local_weight irrelevant (its local_loss is 0); others use 1.0
            agg = multiseed(lambda s: run_interleaved(
                NAN_Classifier(in_dim, 128, 10, arch=a).to(DEVICE),
                tr, te, xb, epochs=10, local_weight=1.0), range(5), in_dim, 128, 10)
            summary[a] = {"acc": agg["acc"]["mean"][-1], "acc_sem": agg["acc"]["sem"][-1],
                          "eff_rank": agg["eff_rank"]["mean"][-1],
                          "dec_sim": agg["dec_sim"]["mean"][-1]}
            v = summary[a]
            ds = "n/a" if v["dec_sim"] < 0 else f"{v['dec_sim']:.3f}"
            print(f"  {a:10} {v['acc']:>8.2f} {v['acc_sem']:>7.2f} {v['eff_rank']:>9.1f} {ds:>8}")
        save_json(summary, "compare_all4.json")

    print(f"\ndone in {time.time()-t0:.1f}s. Outputs in ./{OUT}/")

device=cpu mode=demo arch=static


100%|██████████| 9.91M/9.91M [00:00<00:00, 60.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.79MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 15.6MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.96MB/s]


exp_curves [static]: interleaved ...
exp_curves [static]: sequential (frozen, no penalty) ...
exp_curves [static]: sequential (frozen + diversity penalty) ...
  saved experiments/curves_static.json
  saved experiments/curve_acc_static.png
  saved experiments/curve_rank_static.png
exp_curves [spiking]: interleaved ...
exp_curves [spiking]: sequential (frozen, no penalty) ...
exp_curves [spiking]: sequential (frozen + diversity penalty) ...
  saved experiments/curves_spiking.json
  saved experiments/curve_acc_spiking.png
  saved experiments/curve_rank_spiking.png

done in 165.4s. Outputs in ./experiments/


In [ ]:
if __name__ == "__main__":
    MODE = "div_sweep"   # "demo"|"curves"|"div_sweep"|"local_sweep"|"seq_variants"|"compare_arch"|"compare_all4"
    ARCH = "spiking" # "static"(NAN) | "spiking"(spiking NAN) | "ann"(shared decoder) | "nn"(plain baseline)
    t0 = time.time()
    print(f"device={DEVICE} mode={MODE} arch={ARCH}")

    if MODE == "demo":
        # tiny everything, run BOTH NAN archs so directive compliance is visible
        exp_curves(seeds=[0, 1], epochs=2, res=7, nc=10, hidden=64, arch="static")
        exp_curves(seeds=[0, 1], epochs=2, res=7, nc=10, hidden=64, arch="spiking")

    elif MODE == "curves":
        exp_curves(seeds=range(5), epochs=10, arch=ARCH)

    elif MODE == "div_sweep":
        exp_sweep("div", [0, 0.1, 0.5, 1, 2, 5, 10, 20], seeds=range(5), epochs=6, arch=ARCH)

    elif MODE == "local_sweep":
        exp_sweep("local", [0, 0.1, 0.25, 0.5, 1, 2, 4], seeds=range(5), epochs=6, arch=ARCH)

    elif MODE == "seq_variants":
        exp_seq_variants(seeds=range(5), epochs=10, arch=ARCH)

    elif MODE == "compare_arch":
        # same experiment for both NAN neuron models: static vs spiking
        for a in ("static", "spiking"):
            exp_seq_variants(seeds=range(5), epochs=10, arch=a)
            exp_sweep("div", [0, 0.5, 1, 2, 5, 10], seeds=range(5), epochs=6, arch=a)

    elif MODE == "compare_all4":
        # the paper's full architecture comparison: NN vs ANN vs NAN(static) vs
        # NAN(spiking), interleaved training, final accuracy + diversity per arch.
        tr, te, in_dim = get_loaders(resolution=14, num_classes=10)
        xb = next(iter(te))[0].to(DEVICE)
        print(f"\n  {'arch':10} {'acc':>8} {'+-sem':>7} {'eff_rank':>9} {'dec_sim':>8}")
        summary = {}
        for a in ("nn", "ann", "static", "spiking"):
            # nn uses local_weight irrelevant (its local_loss is 0); others use 1.0
            agg = multiseed(lambda s: run_interleaved(
                NAN_Classifier(in_dim, 128, 10, arch=a).to(DEVICE),
                tr, te, xb, epochs=10, local_weight=1.0), range(5), in_dim, 128, 10)
            summary[a] = {"acc": agg["acc"]["mean"][-1], "acc_sem": agg["acc"]["sem"][-1],
                          "eff_rank": agg["eff_rank"]["mean"][-1],
                          "dec_sim": agg["dec_sim"]["mean"][-1]}
            v = summary[a]
            ds = "n/a" if v["dec_sim"] < 0 else f"{v['dec_sim']:.3f}"
            print(f"  {a:10} {v['acc']:>8.2f} {v['acc_sem']:>7.2f} {v['eff_rank']:>9.1f} {ds:>8}")
        save_json(summary, "compare_all4.json")

    print(f"\ndone in {time.time()-t0:.1f}s. Outputs in ./{OUT}/")

device=cpu mode=div_sweep arch=spiking
exp_sweep[div/spiking]: value=0 ...


In [ ]:
if __name__ == "__main__":
    MODE = "compare_arch"   # "demo"|"curves"|"div_sweep"|"local_sweep"|"seq_variants"|"compare_arch"|"compare_all4"
    ARCH = "spiking" # "static"(NAN) | "spiking"(spiking NAN) | "ann"(shared decoder) | "nn"(plain baseline)
    t0 = time.time()
    print(f"device={DEVICE} mode={MODE} arch={ARCH}")

    if MODE == "demo":
        # tiny everything, run BOTH NAN archs so directive compliance is visible
        exp_curves(seeds=[0, 1], epochs=2, res=7, nc=10, hidden=64, arch="static")
        exp_curves(seeds=[0, 1], epochs=2, res=7, nc=10, hidden=64, arch="spiking")

    elif MODE == "curves":
        exp_curves(seeds=range(5), epochs=10, arch=ARCH)

    elif MODE == "div_sweep":
        exp_sweep("div", [0, 0.1, 0.5, 1, 2, 5, 10, 20], seeds=range(5), epochs=6, arch=ARCH)

    elif MODE == "local_sweep":
        exp_sweep("local", [0, 0.1, 0.25, 0.5, 1, 2, 4], seeds=range(5), epochs=6, arch=ARCH)

    elif MODE == "seq_variants":
        exp_seq_variants(seeds=range(5), epochs=10, arch=ARCH)

    elif MODE == "compare_arch":
        # same experiment for both NAN neuron models: static vs spiking
        for a in ("static", "spiking"):
            exp_seq_variants(seeds=range(5), epochs=10, arch=a)
            exp_sweep("div", [0, 0.5, 1, 2, 5, 10], seeds=range(5), epochs=6, arch=a)

    elif MODE == "compare_all4":
        # the paper's full architecture comparison: NN vs ANN vs NAN(static) vs
        # NAN(spiking), interleaved training, final accuracy + diversity per arch.
        tr, te, in_dim = get_loaders(resolution=14, num_classes=10)
        xb = next(iter(te))[0].to(DEVICE)
        print(f"\n  {'arch':10} {'acc':>8} {'+-sem':>7} {'eff_rank':>9} {'dec_sim':>8}")
        summary = {}
        for a in ("nn", "ann", "static", "spiking"):
            # nn uses local_weight irrelevant (its local_loss is 0); others use 1.0
            agg = multiseed(lambda s: run_interleaved(
                NAN_Classifier(in_dim, 128, 10, arch=a).to(DEVICE),
                tr, te, xb, epochs=10, local_weight=1.0), range(5), in_dim, 128, 10)
            summary[a] = {"acc": agg["acc"]["mean"][-1], "acc_sem": agg["acc"]["sem"][-1],
                          "eff_rank": agg["eff_rank"]["mean"][-1],
                          "dec_sim": agg["dec_sim"]["mean"][-1]}
            v = summary[a]
            ds = "n/a" if v["dec_sim"] < 0 else f"{v['dec_sim']:.3f}"
            print(f"  {a:10} {v['acc']:>8.2f} {v['acc_sem']:>7.2f} {v['eff_rank']:>9.1f} {ds:>8}")
        save_json(summary, "compare_all4.json")

    print(f"\ndone in {time.time()-t0:.1f}s. Outputs in ./{OUT}/")